In [1]:
import os
import re
import numpy as np
import pandas as pd
from datetime import datetime
import lightgbm as lgb
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, log_loss

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 180)

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

IN_DIR    = r"../../data/Riot_API/feature/datamart"
OUT_DIR   = r"../../data/Riot_API/feature/datamart/full"
PATH_FEAT = os.path.join(IN_DIR, "dm_match_team_feature.csv")
PATH_CAT  = os.path.join(IN_DIR, "dm_feature_catalog.csv")

print("IN:", PATH_FEAT)
print("IN:", PATH_CAT)
print("OUT_DIR:", OUT_DIR)

# =========================
# Run config (재현성 고정)
# =========================
SEED = 42
TEST_SIZE = 0.2
SPLIT_LABEL = "matchid_random_80_20"  # early와 통일
VARIANT = "deleaked83"

# 모델 파라미터
N_ESTIMATORS = 800
LEARNING_RATE = 0.02

MAX_DEPTH = 4
NUM_LEAVES = 24

SUBSAMPLE = 0.65
COLSAMPLE = 0.35

REG_LAMBDA = 5.0
REG_ALPHA = 1.0

MIN_CHILD_SAMPLES = 500
MIN_CHILD_WEIGHT = 80

# 대시보드에서 사용할 상위 피처 개수
TOP_K_LOCAL = 15
TOPN_GLOBAL = 25

IN: ../../data/Riot_API/feature/datamart\dm_match_team_feature.csv
IN: ../../data/Riot_API/feature/datamart\dm_feature_catalog.csv
OUT_DIR: ../../data/Riot_API/feature/datamart/full


In [2]:
df = pd.read_csv(PATH_FEAT)
cat = pd.read_csv(PATH_CAT)

print("feature dm:", df.shape)
print("catalog:", cat.shape)
display(df.head(3))
display(cat.head(5))

# required keys
req = ["match_id","team_id","win_team"]
missing = [c for c in req if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns in feature DM: {missing}")

# grain checks
dup = df.duplicated(["match_id","team_id"]).sum()
print("Duplicates (match_id, team_id):", int(dup))
assert dup == 0

rows_per_match = df.groupby("match_id").size()
bad = rows_per_match[rows_per_match != 2]
print("Non-2row matches:", len(bad))
assert len(bad) == 0

bad_team = df.loc[~df["team_id"].isin([100,200]), "team_id"].unique()
print("team_id unique:", df["team_id"].unique())
assert len(bad_team) == 0

wt = df["win_team"]
if wt.dtype == "object":
    wt_norm = wt.astype(str).str.lower().map({"true":1,"false":0,"1":1,"0":0})
else:
    wt_norm = wt.astype(int)

win_sum = wt_norm.groupby(df["match_id"]).sum()
bad_win = win_sum[win_sum != 1]
print("Bad matches (win_team sum != 1):", len(bad_win))
assert len(bad_win) == 0

feature dm: (108412, 114)
catalog: (114, 3)


,match_id,platform,team_id,win_team,queue_id,game_version,game_creation,game_duration,assists_diff,atakhan_kills_diff,baron_kills_diff,deaths_diff,dragon_kills_diff,gold_earned_diff,gold_spent_diff,horde_kills_diff,inhibitor_takedowns_diff,kills_diff,riftHerald_kills_diff,turret_plates_taken_diff,turret_takedowns_diff,vision_score_diff,wards_killed_diff,wards_placed_diff,pos_BOT__assists__diff,pos_BOT__ch_kill_participation__diff,pos_BOT__ch_multikills__diff,pos_BOT__ch_solo_kills__diff,pos_BOT__ch_takedowns__diff,pos_BOT__deaths__diff,pos_BOT__dmg_mitigated__diff,pos_BOT__dmg_taken__diff,pos_BOT__dmg_to_champ__diff,pos_BOT__gold_earned__diff,pos_BOT__gold_spent__diff,pos_BOT__jungle_cs__diff,pos_BOT__kills__diff,pos_BOT__lane_cs__diff,pos_BOT__vision_score__diff,pos_BOT__wards_killed__diff,pos_BOT__wards_placed__diff,pos_JNG__assists__diff,pos_JNG__ch_kill_participation__diff,pos_JNG__ch_multikills__diff,pos_JNG__ch_solo_kills__diff,pos_JNG__ch_takedowns__diff,pos_JNG__deaths__diff,pos_JNG__dmg_mitigated__diff,pos_JNG__dmg_taken__diff,pos_JNG__dmg_to_champ__diff,pos_JNG__gold_earned__diff,pos_JNG__gold_spent__diff,pos_JNG__jungle_cs__diff,pos_JNG__kills__diff,pos_JNG__lane_cs__diff,pos_JNG__vision_score__diff,pos_JNG__wards_killed__diff,pos_JNG__wards_placed__diff,pos_MID__assists__diff,pos_MID__ch_kill_participation__diff,pos_MID__ch_multikills__diff,pos_MID__ch_solo_kills__diff,pos_MID__ch_takedowns__diff,pos_MID__deaths__diff,pos_MID__dmg_mitigated__diff,pos_MID__dmg_taken__diff,pos_MID__dmg_to_champ__diff,pos_MID__gold_earned__diff,pos_MID__gold_spent__diff,pos_MID__jungle_cs__diff,pos_MID__kills__diff,pos_MID__lane_cs__diff,pos_MID__vision_score__diff,pos_MID__wards_killed__diff,pos_MID__wards_placed__diff,pos_SUP__assists__diff,pos_SUP__ch_kill_participation__diff,pos_SUP__ch_multikills__diff,pos_SUP__ch_solo_kills__diff,pos_SUP__ch_takedowns__diff,pos_SUP__deaths__diff,pos_SUP__dmg_mitigated__diff,pos_SUP__dmg_taken__diff,pos_SUP__dmg_to_champ__diff,pos_SUP__gold_earned__diff,pos_SUP__gold_spent__diff,pos_SUP__jungle_cs__diff,pos_SUP__kills__diff,pos_SUP__lane_cs__diff,pos_SUP__vision_score__diff,pos_SUP__wards_killed__diff,pos_SUP__wards_placed__diff,pos_TOP__assists__diff,pos_TOP__ch_kill_participation__diff,pos_TOP__ch_multikills__diff,pos_TOP__ch_solo_kills__diff,pos_TOP__ch_takedowns__diff,pos_TOP__deaths__diff,pos_TOP__dmg_mitigated__diff,pos_TOP__dmg_taken__diff,pos_TOP__dmg_to_champ__diff,pos_TOP__gold_earned__diff,pos_TOP__gold_spent__diff,pos_TOP__jungle_cs__diff,pos_TOP__kills__diff,pos_TOP__lane_cs__diff,pos_TOP__vision_score__diff,pos_TOP__wards_killed__diff,pos_TOP__wards_placed__diff,struct__carry_dependency_gold,struct__carry_dependency_damage,struct__gold_concentration_index,struct__damage_concentration_index,struct__kill_concentration_index
0,BR1_3165661977,br1,100,True,420,15.22.724.5161,1763000735820,1985,-3,1.0,1,-10,3,9678,3690,-3,6,9,-1,-2,13,-18,14,-24,-2,-0.158687,0,0,-1,4,8191,12258,10897,2484,1350,4,1,36,-19,0,-13,-1,-0.034884,2,3,3,-7,-2077,-4986,3026,3056,1000,53,4,-5,8,7,-7,-2,0.006840,1,2,4,-5,-18342,-20899,1981,3386,4415,4,6,24,19,5,3,6,-0.098495,-3,3,0,-3,41813,13807,-10920,-3916,-4100,-4,-6,-38,-26,1,-6,-4,-0.147743,1,0,0,1,8125,11770,-598,4668,1025,17,4,29,0,1,-1,0.248070,0.247976,0.203695,0.204361,0.224446
1,BR1_3165661977,br1,200,False,420,15.22.724.5161,1763000735820,1985,3,-1.0,-1,10,-3,-9678,-3690,3,-6,-9,1,2,-13,18,-14,24,2,0.158687,0,0,1,-4,-8191,-12258,-10897,-2484,-1350,-4,-1,-36,19,0,13,1,0.034884,-2,-3,-3,7,2077,4986,-3026,-3056,-1000,-53,-4,5,-8,-7,7,2,-0.006840,-1,-2,-4,5,18342,20899,-1981,-3386,-4415,-4,-6,-24,-19,-5,-3,-6,0.098495,3,-3,0,3,-41813,-13807,10920,3916,4100,4,6,38,26,-1,6,4,0.147743,-1,0,0,-1,-8125,-11770,598,-4668,-1025,-17,-4,-29,0,-1,1,0.286056,0.294512,0.211240,0.226958,0.268166
2,BR1_3200312712,br1,100,False,420,16.2.741.3171,1769979423613,1927,-25,0.0,-1,13,-4,-8471,-3900,3,-4,-13,-1,-15,-14,-58,-5,-23,-17,-0.087108,0,2,-11,4,-2573,6855,24665,792,835,0

,feature_name,group,ml_allowed
0,match_id,META,0
1,platform,META,0
2,team_id,META,0
3,win_team,META,0
4,queue_id,META,0


Duplicates (match_id, team_id): 0
Non-2row matches: 0
team_id unique: [100 200]
Bad matches (win_team sum != 1): 0


In [3]:
# target
y = df["win_team"].astype(int).values

# base feature set (ml_allowed==1)
feature_cols_full = cat.loc[cat["ml_allowed"]==1, "feature_name"].tolist()
missing_feat = [c for c in feature_cols_full if c not in df.columns]
if missing_feat:
    raise ValueError(f"ml_allowed feature missing in df: {missing_feat[:20]}")

print("ml_allowed==1 features:", len(feature_cols_full))

# drop 20 (pos metrics × 5 positions)
drop_pos_metrics = {"ch_multikills","ch_solo_kills","gold_spent","assists"}
drop_pos_cols=[]
for c in feature_cols_full:
    if c.startswith("pos_") and c.endswith("__diff"):
        parts = c.split("__")  # ["pos_TOP", "assists", "diff"]
        if len(parts) >= 3 and parts[1] in drop_pos_metrics:
            drop_pos_cols.append(c)
drop_pos_cols = sorted(set(drop_pos_cols))
print("drop_pos_cols:", len(drop_pos_cols))

# drop 3 proxies
drop_proxy = {"gold_earned_diff","turret_takedowns_diff","inhibitor_takedowns_diff"}

feature_cols = [c for c in feature_cols_full if (c not in drop_pos_cols and c not in drop_proxy)]

expected = len(feature_cols_full) - len(drop_pos_cols) - len(drop_proxy)

assert expected == len(feature_cols), "Feature count mismatch after drops"
assert len(feature_cols) == 83, f"Expected 83 but got {len(feature_cols)}"

# hard check: proxies excluded
for p in drop_proxy:
    assert p not in feature_cols, f"Proxy still in features: {p}"

print("final feature cols:", len(feature_cols))
print("example:", feature_cols[:12])

ml_allowed==1 features: 106
drop_pos_cols: 20
final feature cols: 83
example: ['assists_diff', 'atakhan_kills_diff', 'baron_kills_diff', 'deaths_diff', 'dragon_kills_diff', 'gold_spent_diff', 'horde_kills_diff', 'kills_diff', 'riftHerald_kills_diff', 'turret_plates_taken_diff', 'vision_score_diff', 'wards_killed_diff']


In [4]:
X = df[feature_cols].copy()

match_ids = df["match_id"].unique()
train_m, test_m = train_test_split(match_ids, test_size=TEST_SIZE, random_state=SEED, shuffle=True)

train_mask = df["match_id"].isin(train_m)
test_mask  = df["match_id"].isin(test_m)

X_train = X.loc[train_mask].reset_index(drop=True)
y_train = y[train_mask]
X_test  = X.loc[test_mask].reset_index(drop=True)
y_test  = y[test_mask]

idx_train = df.index[train_mask].values
idx_test  = df.index[test_mask].values

print("train rows:", X_train.shape, "test rows:", X_test.shape)
print("train matches:", len(train_m), "test matches:", len(test_m))
print("train win rate:", y_train.mean(), "test win rate:", y_test.mean())

overlap = len(set(df.loc[idx_train,"match_id"]) & set(df.loc[idx_test,"match_id"]))
print("overlap matches:", overlap)
assert overlap == 0, "Split leakage: same match_id appears in both train and test!"

train rows: (86728, 83) test rows: (21684, 83)
train matches: 43364 test matches: 10842
train win rate: 0.5 test win rate: 0.5
overlap matches: 0


In [5]:
def train_lgbm(X_train, y_train):
    model = lgb.LGBMClassifier(
        n_estimators=N_ESTIMATORS,
        learning_rate=LEARNING_RATE,
        num_leaves=NUM_LEAVES,
        max_depth=MAX_DEPTH,
        min_child_samples=MIN_CHILD_SAMPLES,
        subsample=SUBSAMPLE,
        colsample_bytree=COLSAMPLE,
        reg_lambda=REG_LAMBDA,
        reg_alpha=REG_ALPHA,
        random_state=SEED,
        n_jobs=-1,
        boosting_type="gbdt",
    )
    model.fit(X_train, y_train)
    return model

def train_xgb(X_train, y_train):
    model = xgb.XGBClassifier(
        n_estimators=N_ESTIMATORS,
        learning_rate=LEARNING_RATE,
        max_depth=MAX_DEPTH,
        min_child_weight=MIN_CHILD_WEIGHT,
        subsample=SUBSAMPLE,
        colsample_bytree=COLSAMPLE,
        reg_lambda=REG_LAMBDA,
        reg_alpha=REG_ALPHA,
        random_state=SEED,
        n_jobs=-1,
        eval_metric="logloss",
        tree_method="hist",
    )
    model.fit(X_train, y_train)
    return model

def eval_model(model, X_train, y_train, X_test, y_test):
    p_train = model.predict_proba(X_train)[:, 1]
    p_test  = model.predict_proba(X_test)[:, 1]
    return {
        "auc_train": roc_auc_score(y_train, p_train),
        "auc_test": roc_auc_score(y_test, p_test),
        "logloss_train": log_loss(y_train, p_train),
        "logloss_test": log_loss(y_test, p_test),
        "p_train": p_train,
        "p_test": p_test,
    }

In [6]:
import shap

def make_explainer(model):
    return shap.TreeExplainer(model, model_output="raw")  # raw=logit 공간

def unwrap_shap(shap_values):
    if isinstance(shap_values, list):
        return shap_values[1]
    return shap_values

def unwrap_base(expected_value):
    if isinstance(expected_value, (list, np.ndarray)):
        return float(expected_value[1] if len(expected_value) > 1 else expected_value[0])
    return float(expected_value)

In [7]:
feature_group_map = dict(zip(cat["feature_name"], cat["group"]))

def metric_group(feature_name: str) -> str:
    f = feature_name.lower()

    if f.startswith("struct__"):
        return "STRUCT"
    if any(x in f for x in ["dragon","baron","herald","horde","atakhan"]):
        return "OBJECTIVE"
    if any(x in f for x in ["turret","plate","inhibitor"]):
        return "STRUCTURE"
    if any(x in f for x in ["vision","ward"]):
        return "VISION"
    if any(x in f for x in ["dmg","damage","taken","mitigated"]):
        return "DAMAGE"
    if any(x in f for x in ["gold","lane_cs","jungle_cs","cs"]):
        return "ECON"
    if any(x in f for x in ["kills","deaths","assists","takedowns","participation"]):
        return "COMBAT"
    return "OTHER"

In [8]:
def build_global_shap(model_name, base_logit, base_prob, phi, feat_names):
    mean_abs_shap = np.mean(np.abs(phi), axis=0)
    mean_signed_shap = np.mean(phi, axis=0)

    dp = sigmoid(base_logit + phi) - base_prob
    mean_abs_dp = np.mean(np.abs(dp), axis=0)
    mean_signed_dp = np.mean(dp, axis=0)

    out = pd.DataFrame({
        "model_name": model_name,
        "variant": VARIANT,
        "split": SPLIT_LABEL,
        "seed": SEED,
        "n_estimators": N_ESTIMATORS,
        "base_logit": base_logit,
        "base_prob": base_prob,
        "feature_name": feat_names,
        "feature_group": [feature_group_map.get(f, "UNKNOWN") for f in feat_names],
        "metric_group": [metric_group(f) for f in feat_names],
        "mean_abs_shap_logit": mean_abs_shap,
        "mean_signed_shap_logit": mean_signed_shap,
        "mean_abs_delta_p": mean_abs_dp,
        "mean_signed_delta_p": mean_signed_dp,
    }).sort_values("mean_abs_shap_logit", ascending=False).reset_index(drop=True)

    out["rank"] = np.arange(1, len(out) + 1)
    return out

def build_group_shap(model_name, base_logit, base_prob, phi, feat_cols_used):
    # Group SHAP: TOP/JNG/MID/BOT/SUP + TEAM_DIFF + STRUCT
    groups = ["TOP","JNG","MID","BOT","SUP","TEAM_DIFF","STRUCT"]

    group_to_idx={}
    for g in groups:
        feats_g = cat.loc[cat["group"]==g, "feature_name"].tolist()
        idx = [i for i, f in enumerate(feat_cols_used) if f in feats_g]
        group_to_idx[g] = idx

    rows=[]
    for g, idxs in group_to_idx.items():
        if len(idxs) == 0:
            continue
        g_logit = phi[:, idxs].sum(axis=1)
        g_dp = sigmoid(base_logit + g_logit) - base_prob

        rows.append({
            "model_name": model_name,
            "variant": VARIANT,
            "split": SPLIT_LABEL,
            "seed": SEED,
            "n_estimators": N_ESTIMATORS,
            "base_logit": base_logit,
            "base_prob": base_prob,
            "group": g,
            "n_features_in_group": len(idxs),
            "mean_abs_shap_logit": float(np.mean(np.abs(g_logit))),
            "mean_signed_shap_logit": float(np.mean(g_logit)),
            "mean_abs_delta_p": float(np.mean(np.abs(g_dp))),
            "mean_signed_delta_p": float(np.mean(g_dp)),
        })

    out = pd.DataFrame(rows).sort_values("mean_abs_shap_logit", ascending=False).reset_index(drop=True)
    out["rank"] = np.arange(1, len(out) + 1)
    return out

def build_local_shap(model_name, base_logit, base_prob, phi, feat_names, id_df, pred_prob, top_k=15):
    rows=[]
    for i in range(phi.shape[0]):
        row_phi = phi[i, :]
        abs_phi = np.abs(row_phi)
        top_idx = np.argsort(-abs_phi)[:top_k]

        top_phi = row_phi[top_idx]
        top_dp = sigmoid(base_logit + top_phi) - base_prob

        tmp = pd.DataFrame({
            "model_name": model_name,
            "variant": VARIANT,
            "split": SPLIT_LABEL,
            "seed": SEED,
            "n_estimators": N_ESTIMATORS,
            "match_id": id_df.loc[i, "match_id"],
            "team_id": id_df.loc[i, "team_id"],
            "pred_prob_full": float(pred_prob[i]),
            "feature_name": feat_names[top_idx],
            "feature_group": [feature_group_map.get(f, "UNKNOWN") for f in feat_names[top_idx]],
            "metric_group": [metric_group(f) for f in feat_names[top_idx]],
            "shap_logit": top_phi,
            "abs_shap_logit": np.abs(top_phi),
            "delta_p": top_dp,
            "abs_delta_p": np.abs(top_dp),
        }).sort_values("abs_shap_logit", ascending=False).reset_index(drop=True)

        tmp["rank_in_row"] = np.arange(1, len(tmp) + 1)
        rows.append(tmp)

    return pd.concat(rows, ignore_index=True)

In [9]:
feat_names = np.array(feature_cols)

# test ids (join key)
id_df_test = df.loc[idx_test, ["match_id","team_id"]].reset_index(drop=True)

global_all=[]
group_all=[]
local_all=[]
metrics_rows=[]

for model_name in ["lightgbm","xgboost"]:
    try:
        if model_name == "lightgbm":
            model = train_lgbm(X_train, y_train)
        else:
            model = train_xgb(X_train, y_train)

        met = eval_model(model, X_train, y_train, X_test, y_test)
        print(f"\n[{model_name}] AUC train={met['auc_train']:.4f}, test={met['auc_test']:.4f} | "
              f"Logloss train={met['logloss_train']:.4f}, test={met['logloss_test']:.4f}")

        explainer = make_explainer(model)
        shap_test = unwrap_shap(explainer.shap_values(X_test))
        base_logit = unwrap_base(explainer.expected_value)
        base_prob = float(sigmoid(base_logit))

        gdf = build_global_shap(model_name, base_logit, base_prob, shap_test, feat_names)
        grdf = build_group_shap(model_name, base_logit, base_prob, shap_test, feature_cols)
        ldf = build_local_shap(model_name, base_logit, base_prob, shap_test, feat_names, id_df_test, met["p_test"], top_k=TOP_K_LOCAL)

        global_all.append(gdf)
        group_all.append(grdf)
        local_all.append(ldf)

        metrics_rows.append({
            "model_name": model_name,
            "variant": VARIANT,
            "split": SPLIT_LABEL,
            "seed": SEED,
            "n_estimators": N_ESTIMATORS,
            "learning_rate": LEARNING_RATE,
            "n_features": int(X_train.shape[1]),
            "n_train_rows": int(X_train.shape[0]),
            "n_test_rows": int(X_test.shape[0]),
            "n_train_matches": int(len(train_m)),
            "n_test_matches": int(len(test_m)),
            "auc_train": met["auc_train"],
            "auc_test": met["auc_test"],
            "logloss_train": met["logloss_train"],
            "logloss_test": met["logloss_test"],
            "base_logit": base_logit,
            "base_prob": base_prob,
        })

    except Exception as e:
        print(f"[{model_name}] ERROR:", repr(e))

global_df = pd.concat(global_all, ignore_index=True)
group_df  = pd.concat(group_all, ignore_index=True)
local_df  = pd.concat(local_all, ignore_index=True)
metrics_df = pd.DataFrame(metrics_rows)

# ---- Save core outputs ----
OUT_GLOBAL  = os.path.join(OUT_DIR, "dm_match_team_shap_global_full.csv")
OUT_GROUP   = os.path.join(OUT_DIR, "dm_match_team_shap_group_full.csv")
OUT_METRICS = os.path.join(OUT_DIR, "dm_model_metrics_full.csv")

global_df.to_csv(OUT_GLOBAL, index=False)
group_df.to_csv(OUT_GROUP, index=False)
metrics_df.to_csv(OUT_METRICS, index=False)

print("\nSaved core:")
print(" -", OUT_GLOBAL, global_df.shape)
print(" -", OUT_GROUP, group_df.shape)
print(" -", OUT_METRICS, metrics_df.shape)

display(metrics_df)

[LightGBM] [Info] Number of positive: 43364, number of negative: 43364
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.015817 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 13022
[LightGBM] [Info] Number of data points in the train set: 86728, number of used features: 83
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further spl

c:\Users\andor\AppData\Local\Programs\Python\Python311\Lib\site-packages\shap\explainers\_tree.py:587: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray
  warnings.warn(



[xgboost] AUC train=0.9975, test=0.9964 | Logloss train=0.0694, test=0.0800

Saved core:
 - ../../data/Riot_API/feature/datamart/full\dm_match_team_shap_global_full.csv (166, 15)
 - ../../data/Riot_API/feature/datamart/full\dm_match_team_shap_group_full.csv (14, 14)
 - ../../data/Riot_API/feature/datamart/full\dm_model_metrics_full.csv (2, 17)


,model_name,variant,split,seed,n_estimators,learning_rate,n_features,n_train_rows,n_test_rows,n_train_matches,n_test_matches,auc_train,auc_test,logloss_train,logloss_test,base_logit,base_prob
0,lightgbm,deleaked83,matchid_random_80_20,42,800,0.02,83,86728,21684,43364,10842,0.998008,0.996563,0.064166,0.078114,-0.008508,0.497873
1,xgboost,deleaked83,matchid_random_80_20,42,800,0.02,83,86728,21684,43364,10842,0.997547,0.996424,0.069439,0.080025,0.001007,0.500252


In [10]:
# join 가능한 meta subset (feature DM에 있는 것만)
meta_for_join = [c for c in [
    "match_id","team_id","platform","queue_id","game_version","game_creation","game_duration","win_team"
] if c in df.columns]

meta_df = df.loc[idx_test, meta_for_join].reset_index(drop=True)

local_with_meta = local_df.merge(meta_df, on=["match_id","team_id"], how="left")

OUT_LOCAL_META = os.path.join(OUT_DIR, "dm_match_team_shap_local_full_with_meta.csv")
local_with_meta.to_csv(OUT_LOCAL_META, index=False)

print("Saved local with meta:")
print(" -", OUT_LOCAL_META, local_with_meta.shape)

Saved local with meta:
 - ../../data/Riot_API/feature/datamart/full\dm_match_team_shap_local_full_with_meta.csv (650520, 22)


In [11]:
# 1) local extreme 비율 체크
for model_name in local_df["model_name"].unique():
    key = local_df[local_df["model_name"]==model_name].groupby(["match_id","team_id"], as_index=False)["pred_prob_full"].first()
    per_match = key.groupby("match_id")["pred_prob_full"].agg(["count","min","max"])
    extreme_rate = ((per_match["max"]>0.995) & (per_match["min"]<0.005)).mean()
    print(f"{model_name}: test_matches={len(per_match)}, extreme_rate={extreme_rate:.1%}")

# 2) Group SHAP 포지션만 보기 (Tab3 주력)
for model_name in group_df["model_name"].unique():
    sub = group_df[(group_df["model_name"]==model_name) & (group_df["group"].isin(["TOP","JNG","MID","BOT","SUP"]))].sort_values("rank")
    print("\n", model_name, "position ranks:")
    print(sub[["rank","group","mean_abs_delta_p","n_features_in_group"]].to_string(index=False))

lightgbm: test_matches=10842, extreme_rate=48.4%
xgboost: test_matches=10842, extreme_rate=45.4%

 lightgbm position ranks:
 rank group  mean_abs_delta_p  n_features_in_group
    2   SUP          0.237605                   13
    3   TOP          0.229742                   13
    4   JNG          0.220956                   13
    5   MID          0.219400                   13
    6   BOT          0.166357                   13

 xgboost position ranks:
 rank group  mean_abs_delta_p  n_features_in_group
    2   SUP          0.226133                   13
    3   TOP          0.219478                   13
    4   MID          0.206801                   13
    5   JNG          0.202574                   13
    6   BOT          0.169355                   13
